The Catch: Resolution vs. Batch Size
Even with a solid 6GB of VRAM, 512x512 images are massive. To prevent the GPU memory from crashing, we are forced to drop the BATCH_SIZE from 16 down to 4.

Here is the problem: neural networks use a mathematical trick called Batch Normalization. If you only feed the AI 4 images at a time, the statistical average is incredibly noisy. The AI gets confused, the loss bounces up and down erratically, and your accuracy actually drops despite the higher resolution.

The Solution: Gradient Accumulation
We can have our cake and eat it too. We will use a technique called Gradient Accumulation.
Here is how it tricks your GPU into being a supercomputer:

We feed 4 high-res images into the VRAM.

The AI calculates the calculus (loss and gradients) for those 4 images, but does not update its weights yet.

It clears the images from VRAM, but remembers the math.

It repeats this 4 times (4 batches of 4 images = 16 total images).

Finally, it combines the math from all 16 images and updates the weights all at once!

You get the extreme detail of 512x512 images, and the mathematical stability of a batch size of 16, without blowing up your 6GB graphics card.


In [2]:
import os
import pandas as pd
import numpy as np
import cv2
import zipfile
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import timm
from tqdm import tqdm

# ==========================================
# 1. THE 512x512 CONFIGURATION
# ==========================================
# Look for the CSV in the current project folder
CSV_PATH = r'train.csv'          

# Point directly to your master Kaggle zip file in the D:\RSNA folder
ZIP_PATH = r'D:\RSNA\rsna-knee-abnormality-detection.zip'   

IMG_SIZE = 512                  
BATCH_SIZE = 4                  
ACCUMULATION_STEPS = 4          
EPOCHS = 20                     
LEARNING_RATE = 0.001
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Hardware Engine locked to: {DEVICE}")
print(f"Image Resolution set to: {IMG_SIZE}x{IMG_SIZE}")

train_df = pd.read_csv(CSV_PATH)
train_df = train_df.fillna(0.0) 
print("Labels successfully loaded into memory!")

Hardware Engine locked to: cuda
Image Resolution set to: 512x512
Labels successfully loaded into memory!


In [3]:
print(train_df.columns.tolist())

['StudyInstanceUID', 'Report', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']


In [4]:
# ==========================================
# 2. DATASET PIPELINE (DICOM NATIVE)
# ==========================================
import pydicom
import io

class RSNAZipDataset(Dataset):
    def __init__(self, dataframe, zip_path, target_size=IMG_SIZE):
        self.dataframe = dataframe
        self.zip_path = zip_path
        self.target_size = target_size
        self.archive = zipfile.ZipFile(self.zip_path, 'r')
        self.all_files = set(self.archive.namelist())
        
        self.disease_columns = [
            'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 
            'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 
            'Synovitis', "Baker's", 'Contusion', 'Fracture'
        ] 
        
        self.patients = self.dataframe['StudyInstanceUID'].unique()

    def __len__(self):
        return len(self.patients)

    def __getitem__(self, idx):
        patient_id = self.patients[idx]
        patient_data = self.dataframe[self.dataframe['StudyInstanceUID'] == patient_id].iloc[0]
        labels = patient_data[self.disease_columns].values.astype(np.float32)
        
        patient_files = [f for f in self.all_files if f"/{patient_id}/" in f and f.endswith('.dcm')]
        if not patient_files:
            patient_files = [f for f in self.all_files if f.startswith(f"{patient_id}/") and f.endswith('.dcm')]
            
        first_image_path = patient_files[0]
        
        # --- THE DICOM FIX ---
        dicom_bytes = self.archive.read(first_image_path)
        dicom_io = io.BytesIO(dicom_bytes)
        
        # Use pydicom to properly parse the medical file
        dcm = pydicom.dcmread(dicom_io)
        image = dcm.pixel_array.astype(np.float32)
        
        # Medical images have wild pixel ranges. We MUST normalize them to 0-255!
        image = (image - np.min(image)) / (np.max(image) - np.min(image) + 1e-8)
        image = (image * 255).astype(np.uint8)
        
        # Now OpenCV can safely resize the normalized image
        image = cv2.resize(image, (self.target_size, self.target_size))
        
        image_tensor = torch.tensor(image, dtype=torch.float32).unsqueeze(0) / 255.0
        labels_tensor = torch.tensor(labels, dtype=torch.float32)
        
        return {'image': image_tensor, 'labels': labels_tensor}

my_dataset = RSNAZipDataset(dataframe=train_df, zip_path=ZIP_PATH)
train_loader = DataLoader(my_dataset, batch_size=BATCH_SIZE, shuffle=True)
print("Maximum Resolution Data Engine Online.")

Maximum Resolution Data Engine Online.


In [5]:
# ==========================================
# 3. NEURAL NETWORK ARCHITECTURE
# ==========================================
class RSNAKneeModel(nn.Module):
    def __init__(self, model_name='efficientnet_b0', num_classes=12):
        super().__init__()
        self.backbone = timm.create_model(
            model_name, 
            pretrained=True, 
            in_chans=1, 
            num_classes=0 
        )
        self.head = nn.Linear(self.backbone.num_features, num_classes)
        
    def forward(self, x):
        features = self.backbone(x)
        output = self.head(features)
        return output

# We dynamically grab the length of your disease_columns list (12)
model = RSNAKneeModel(num_classes=len(my_dataset.disease_columns)).to(DEVICE)
print("Brain built and moved to VRAM.")

Brain built and moved to VRAM.


In [6]:
# ==========================================
# 4. KAGGLE MASTER TRAINING LOOP (WITH CHECKPOINTS)
# ==========================================
criterion = nn.BCEWithLogitsLoss() 
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scaler = torch.amp.GradScaler('cuda')

print(f"Starting {EPOCHS}-Epoch Maximum Precision Engine...")

for epoch in range(EPOCHS):
    model.train() 
    running_loss = 0.0
    optimizer.zero_grad() 
    
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
    
    for i, batch in enumerate(progress_bar):
        images = batch['image'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        
        with torch.amp.autocast('cuda'):
            predictions = model(images)
            loss = criterion(predictions, labels)
            loss = loss / ACCUMULATION_STEPS 
        
        scaler.scale(loss).backward()
        
        if (i + 1) % ACCUMULATION_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad() 
            
        running_loss += (loss.item() * ACCUMULATION_STEPS)
        progress_bar.set_postfix({'Loss': f"{(loss.item() * ACCUMULATION_STEPS):.4f}"})
        
    epoch_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1} Completed! Average Loss: {epoch_loss:.4f}")
    
    # --- THE CHECKPOINT SAVER ---
    # This saves a new file for every single epoch (e.g., rsna_model_epoch_1.pth)
    checkpoint_path = f'rsna_model_epoch_{epoch+1}.pth'
    torch.save(model.state_dict(), checkpoint_path)
    print(f"--> Checkpoint safely locked to disk: {checkpoint_path}\n")

print("Master Training Run Complete.")

Starting 20-Epoch Maximum Precision Engine...


Epoch 1/20: 100%|██████████| 1102/1102 [23:50<00:00,  1.30s/it, Loss=0.0022]


Epoch 1 Completed! Average Loss: 0.0397
--> Checkpoint safely locked to disk: rsna_model_epoch_1.pth



Epoch 2/20: 100%|██████████| 1102/1102 [21:58<00:00,  1.20s/it, Loss=0.0056]


Epoch 2 Completed! Average Loss: 0.0313
--> Checkpoint safely locked to disk: rsna_model_epoch_2.pth



...

Epoch 18 Completed! Average Loss: 0.0208
--> Checkpoint safely locked to disk: rsna_model_epoch_18.pth



Epoch 19/20: 100%|██████████| 1102/1102 [22:01<00:00,  1.20s/it, Loss=0.0002]


Epoch 19 Completed! Average Loss: 0.0153
--> Checkpoint safely locked to disk: rsna_model_epoch_19.pth



Epoch 20/20: 100%|██████████| 1102/1102 [22:01<00:00,  1.20s/it, Loss=0.0012]

Epoch 20 Completed! Average Loss: 0.0135
--> Checkpoint safely locked to disk: rsna_model_epoch_20.pth

Master Training Run Complete.
